Configurar el entorno y definir los candidatos ficticios

In [5]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.spatial.distance import cosine

# 1. Definimos una base de datos de candidatos (Cartas de presentación)
# Nota: El modelo espera textos en inglés como tu dataset original
candidatos_db = [
    {
        "Nombre": "Elena Martínez",
        "Puesto": "Data Scientist",
        "Texto": "I am a deeply organized and methodical person. I love analyzing every single detail of data and documentation before making a decision. I prefer working in quiet environments where I can focus, and I am highly committed to my deadlines."
    },
    {
        "Nombre": "Carlos Vega",
        "Puesto": "Project Manager",
        "Texto": "I absolutely love leading teams and organizing social events! I am highly energetic, always positive, and I adapt instantly to changes. I don't mind chaotic environments as long as we can brainstorm together out loud."
    },
    {
        "Nombre": "Sofía Rossi",
        "Puesto": "UX Designer",
        "Texto": "Art, philosophy, and abstract ideas are my passion. I always look for unconventional solutions and alternative designs. I am highly empathetic with users, feel deep compassion for others, and value harmony in the team above everything else."
    },
    {
        "Nombre": "David Cho",
        "Puesto": "Security Engineer",
        "Texto": "I tend to worry a lot about potential system failures, which makes me extremely cautious and vigilant. I am quiet, rarely speak in public unless necessary, but I am incredibly meticulous. I follow strict protocols to avoid risks."
    },
    {
        "Nombre": "Laura Brooks",
        "Puesto": "Sales Director",
        "Texto": "I am a natural born persuader and love public speaking. I feel very comfortable under heavy pressure and aggressive targets. I am competitive, assertive, and enjoy being the center of attention in business meetings."
    }
]

df_candidatos = pd.DataFrame(candidatos_db)
print(f"Base de datos cargada con {len(df_candidatos)} perfiles listos para análisis.")

Base de datos cargada con 5 perfiles listos para análisis.


Cargar el modelo RoBERTa entrenado y extraer la Huella Psicométrica

In [7]:
# 1. Rutas de tu Google Drive
from google.colab import drive
drive.mount('/content/drive')

# CORRECCIÓN: Apuntamos exactamente a la subcarpeta que contiene los pesos ganadores
ruta_modelo = '/content/drive/MyDrive/TFM_Personalidad/Mi_Modelo_Entrenado_Final/checkpoint-333'

print("Cargando tu RoBERTa personalizado desde Drive...")
tokenizador = AutoTokenizer.from_pretrained("roberta-base")
# Cargamos el modelo apuntando a la carpeta correcta
modelo = AutoModelForSequenceClassification.from_pretrained(ruta_modelo)
modelo.eval() # Modo evaluación (apaga el dropout para que las predicciones sean estables)

rasgos = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']
vectores_psicometricos = []

# 2. Extracción de vectores en el espacio latente de la personalidad
print("Analizando psicológicamente a los candidatos...")
for texto in df_candidatos['Texto']:
    inputs = tokenizador(texto, return_tensors="pt", padding=True, truncation=True, max_length=512)

    with torch.no_grad():
        outputs = modelo(**inputs)
        logits = outputs.logits
        # Usamos Sigmoide para obtener probabilidades continuas entre 0.0 y 1.0
        probabilidades = torch.sigmoid(logits).numpy()[0]
        vectores_psicometricos.append(probabilidades)

# Añadimos los vectores resultantes al DataFrame
df_candidatos['Vector_Personalidad'] = vectores_psicometricos
print("\n¡Análisis completado! Cada candidato tiene ahora una firma vectorial de 5 dimensiones.")

Mounted at /content/drive
Cargando tu RoBERTa personalizado desde Drive...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Analizando psicológicamente a los candidatos...

¡Análisis completado! Cada candidato tiene ahora una firma vectorial de 5 dimensiones.


El Motor de Recomendación Vectorial

In [8]:
def buscar_candidato_ideal(perfil_buscado_dict):
    # Convertimos el diccionario de RR.HH. en un vector matemático ordenado
    vector_ideal = np.array([perfil_buscado_dict[rasgo] for rasgo in rasgos])

    puntuaciones_similitud = []

    for _, row in df_candidatos.iterrows():
        vector_candidato = row['Vector_Personalidad']

        # Calculamos la Similitud del Coseno (1 - distancia de coseno)
        # 1.0 significa vectores idénticos, 0.0 significa vectores ortogonales
        similitud = 1 - cosine(vector_ideal, vector_candidato)
        puntuaciones_similitud.append(round(similitud * 100, 2))

    # Creamos una tabla de resultados ordenada por afinidad
    df_ranking = df_candidatos.copy()
    df_ranking['Afinidad_Cultural (%)'] = puntuaciones_similitud
    df_ranking = df_ranking.sort_values(by='Afinidad_Cultural (%)', ascending=False)

    return df_ranking[['Nombre', 'Puesto', 'Afinidad_Cultural (%)', 'Vector_Personalidad']]

print("Motor de recomendación vectorial compilado y listo.")

Motor de recomendación vectorial compilado y listo.


Simulación de Casos de Uso Reales

In [ ]:
print("======================================================================")
print("CASO DE USO 1: RR.HH. Busca un líder para un proyecto crítico y caótico")
print("Requisitos: Muy Extrovertido (1.0), resistente al estrés/Bajo Neuroticismo (0.0), Alta Responsabilidad (1.0)")
print("======================================================================")

perfil_leader = {
    'Extroversion': 1.0, 'Neuroticism': 0.0, 'Agreeableness': 0.5, 'Conscientiousness': 1.0, 'Openness': 0.7
}
ranking_leader = buscar_candidato_ideal(perfil_leader)
print(ranking_leader.to_string(index=False))


print("\n\n======================================================================")
print("CASO DE USO 2: RR.HH. Busca un Diseñador Creativo e Innovador")
print("Requisitos: Máxima Apertura a la experiencia (1.0) y muy empático/Amable (1.0)")
print("======================================================================")

perfil_creative = {
    'Extroversion': 0.5, 'Neuroticism': 0.3, 'Agreeableness': 1.0, 'Conscientiousness': 0.4, 'Openness': 1.0
}
ranking_creative = buscar_candidato_ideal(perfil_creative)
print(ranking_creative.to_string(index=False))

CASO DE USO 1: RR.HH. Busca un líder para un proyecto crítico y caótico
Requisitos: Muy Extrovertido (1.0), resistente al estrés/Bajo Neuroticismo (0.0), Alta Responsabilidad (1.0)
        Nombre            Puesto  Afinidad_Cultural (%)                                        Vector_Personalidad
   Carlos Vega   Project Manager                  89.39  [0.5616917, 0.4617249, 0.58022493, 0.5841926, 0.49880648]
  Laura Brooks    Sales Director                  88.39  [0.53517747, 0.4751692, 0.53091615, 0.53927183, 0.508632]
Elena Martínez    Data Scientist                  87.10 [0.4627305, 0.4772374, 0.53987527, 0.55635846, 0.50574344]
     David Cho Security Engineer                  86.88   [0.513529, 0.49419174, 0.5665967, 0.5331445, 0.48980728]
   Sofía Rossi       UX Designer                  85.99  [0.46374673, 0.501543, 0.5083334, 0.51698804, 0.58624154]


CASO DE USO 2: RR.HH. Busca un Diseñador Creativo e Innovador
Requisitos: Máxima Apertura a la experiencia (1.0) y muy empático

Instalar y Configurar la Conexión con Llama 3 (Groq)

In [1]:
!pip install -q groq

import getpass
from groq import Groq

# Pedimos la clave de forma segura para no dejarla escrita en el código
print("Introduce tu API Key de Groq:")
GROQ_KEY = getpass.getpass()

# Inicializamos el cliente "Analista"
cliente_groq = Groq(api_key=GROQ_KEY)
modelo_llama = "llama-3.3-70b-versatile"

print("¡Agente Analista (Llama 3.3) conectado a la nube con éxito!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.7 MB/s eta 0:00:00
Introduce tu API Key de Groq:
··········
¡Agente Analista (Llama 3.3) conectado a la nube con éxito!


Cálculo de Deltas

In [2]:
import numpy as np

def calcular_deltas_psicologicos(vector_candidato, vector_ideal):
    rasgos = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']

    # Calculamos la diferencia matemática: Ideal - Candidato
    # Si es positivo, al candidato "le falta" ese rasgo. Si es negativo, "le sobra".
    diferencias = np.array(vector_ideal) - np.array(vector_candidato)

    # Buscamos dónde cojea más (mayor diferencia positiva) y dónde destaca (mayor valor absoluto o alineación)
    indice_debilidad = np.argmax(diferencias)

    # Para la fortaleza, buscamos el rasgo donde el candidato puntúa más alto por su naturaleza
    indice_fortaleza = np.argmax(vector_candidato)

    return {
        "rasgo_debil": rasgos[indice_debilidad],
        "delta_debilidad": round(diferencias[indice_debilidad], 2),
        "rasgo_fuerte": rasgos[indice_fortaleza],
        "valor_fortaleza": round(vector_candidato[indice_fortaleza], 2)
    }

print("Motor de cálculo de Deltas vectoriales preparado.")

Motor de cálculo de Deltas vectoriales preparado.


El Prompting del Flujo Agéntico (PAG)

In [3]:
def generar_dossier_entrevista(nombre, puesto, texto_carta, vector_candidato, vector_ideal):
    # 1. Calculamos las matemáticas subyacentes
    analisis = calcular_deltas_psicologicos(vector_candidato, vector_ideal)

    # 2. Construimos el System Prompt Dinámico
    system_prompt = f"""
    Eres un Reclutador Senior y Psicólogo Experto en perfiles tecnológicos.
    Tu objetivo es diseñar una Entrevista Conductual Estructurada para un candidato.

    DATOS MATEMÁTICOS DEL MODELO DE CLASIFICACIÓN (ROBERTA):
    - Puesto a cubrir: {puesto}
    - Fortaleza Principal del Candidato: {analisis['rasgo_fuerte']} (Nivel: {analisis['valor_fortaleza']})
    - Debilidad Crítica respecto al puesto: Le falta {analisis['rasgo_debil']} (Distancia Delta: {analisis['delta_debilidad']})

    CARTA DE PRESENTACIÓN DEL CANDIDATO:
    "{texto_carta}"

    INSTRUCCIONES ESTRUCTURALES:
    Genera un DOSSIER DE ENTREVISTA de exactamente 5 fases para este candidato.
    Usa tono profesional pero directo. Las preguntas deben ser para leer en voz alta.
    ESTRUCTURA OBLIGATORIA:
    1. FASE 1: Rompehielos (Apóyate en su fortaleza para que se relaje).
    2. FASE 2: Test de Tensión (Ataca directamente su debilidad crítica detectada).
    3. FASE 3: Escenario Crítico (Un roleplay rápido relacionado con el puesto).
    4. FASE 4: Búsqueda de Contradicciones (Busca grietas en su carta de presentación).
    5. FASE 5: Banderas Rojas (Dile al entrevistador en qué lenguaje no verbal o respuestas evasivas fijarse).

    Devuelve ÚNICAMENTE el dossier formateado en Markdown.
    """

    # 3. Llamada a la API de Groq
    try:
        respuesta = cliente_groq.chat.completions.create(
            messages=[{"role": "system", "content": system_prompt}],
            model=modelo_llama,
            temperature=0.3, # Baja temperatura para que sea clínico y no excesivamente creativo
            max_tokens=1500
        )
        return respuesta.choices[0].message.content.strip()
    except Exception as e:
        return f"Error en la API de Groq: {e}"

print("Función del Agente Autónomo compilada y lista.")

Función del Agente Autónomo compilada y lista.


In [9]:
# 1. Definimos el perfil ideal extremo (Lo contrario a David)
perfil_buscado = {
    'Extroversion': 0.9, 'Neuroticism': 0.1, 'Agreeableness': 0.5, 'Conscientiousness': 0.8, 'Openness': 0.5
}
vector_ideal_prueba = [perfil_buscado[r] for r in rasgos]

# 2. Rescatamos los datos de David Cho de nuestro DataFrame
datos_david = df_candidatos[df_candidatos['Nombre'] == 'David Cho'].iloc[0]
vector_david = datos_david['Vector_Personalidad']

# 3. Generamos el dossier
print(f"Generando Dossier Híbrido para {datos_david['Nombre']}...")
print("Modelo Local (RoBERTa) calculando vectores -> Modelo Cloud (Llama 3.3) redactando protocolo...\n")
print("="*80)

dossier_final = generar_dossier_entrevista(
    nombre=datos_david['Nombre'],
    puesto=datos_david['Puesto'],
    texto_carta=datos_david['Texto'],
    vector_candidato=vector_david,
    vector_ideal=vector_ideal_prueba
)

print(dossier_final)
print("="*80)

Generando Dossier Híbrido para David Cho...
Modelo Local (RoBERTa) calculando vectores -> Modelo Cloud (Llama 3.3) redactando protocolo...

# Dossier de Entrevista para Security Engineer
## FASE 1: Rompehielos
Para comenzar, nos gustaría saber más sobre su enfoque en el trabajo. ¿Podría contarnos sobre una situación en la que su cautela y vigilancia ayudaron a prevenir un problema de seguridad? ¿Cómo se sintió al haber evitado ese riesgo?

## FASE 2: Test de Tensión
Considerando su tendencia a ser quieto y no hablar en público a menos que sea necesario, ¿cómo manejaría una situación en la que necesitara comunicar un problema de seguridad crítico a un equipo grande y diverso, incluyendo a personas que no están familiarizadas con la jerga técnica? ¿Podría describir su estrategia para asegurarse de que su mensaje sea claro y efectivo?

## FASE 3: Escenario Crítico
Imagínese que está trabajando como Security Engineer y descubre una vulnerabilidad en uno de nuestros sistemas críticos. El eq